# သင်ခန်းစာ ၁၃ - Cognee နဲ့ အေးဂျင့်မှတ်ဉာဏ် Knowledge Graphs


## စတင်ပြင်ဆင်ခြင်း

ဒီ notebook မှာ persistent memory ပါတဲ့ ဉာဏ်ရည်မြင့် **coding assistant** တစ်ခုကို [**Cognee**](https://www.cognee.ai/) knowledge graphs နဲ့ **Microsoft Agent Framework** (MAF) ကို အသုံးပြုပြီး ဘယ်လိုဖန်တီးရမလဲကို ပြသထားပါတယ်။

Cognee က unstructured စာသားကို vector embeddings အားဖြင့် supported ဖြစ်တဲ့ structured, query လုပ်နိုင်တဲ့ knowledge graph အဖြစ် ပြောင်းလဲပေးပါတယ် — သင့် agent ကို ပိုမိုကြီးမားပြီး ဆက်စပ်မှုရှိတဲ့ ရေရှည်မှတ်ဉာဏ် တစ်ခုဆီသို့ ရောက်စေပါတယ်။

### သင်ဘာတွေသင်ယူမှာလဲ
1. **Knowledge Graphs ဖန်တီးခြင်း**: Developer profiles နဲ့ best practices တွေကို structured, query လုပ်နိုင်တဲ့ knowledge အဖြစ် ပြောင်းလဲခြင်း။
2. **Cognee ကို MAF နဲ့ပေါင်းစပ်ခြင်း**: MAF agent တစ်ခုကို Cognee ရဲ့ knowledge graph ကို query လုပ်ခွင့်ပေးဖို့ `@tool` function တွေကို အသုံးပြုခြင်း။
3. **Session-Aware စကားဝိုင်းများ**: တူညီတဲ့ session အတွင်း မေးခွန်းများစွာအကြား context ကို ထိန်းသိမ်းထားခြင်း။
4. **ရေရှည်မှတ်ဉာဏ်**: အရေးကြီးသော knowledge ကို sessions များအနှံ့ တည်တံ့စေပြီး ဆွေးနွေးမှုအသစ်များတွင် ပြန်လည်ယူနိုင်ခြင်း။

### လိုအပ်သော စီစဉ်ချက်များ
- Python 3.9+
- Redis ကို ဒေသတွင်းတွင် စတင်ထားရှိပြီး (`docker run -d -p 6379:6379 redis`) session management အတွက်
- LLM API key တစ်ခု (ဥပမာ OpenAI) — `.env` ထဲတွင် `LLM_API_KEY` ကို သတ်မှတ်ထားရန်
- `.env` ထဲတွင် `CACHING=true` (Cognee sessions အတွက် လိုအပ်သည်)
- Microsoft Foundry project တစ်ခုအနေနဲ့ chat model တပ်ဆင်ပြီး
- `.env` ထဲတွင် `AZURE_AI_PROJECT_ENDPOINT` နဲ့ `AZURE_AI_MODEL_DEPLOYMENT_NAME` ကို သတ်မှတ်ထားရန်
- Azure CLI မှာ အတည်ပြုထားပြီး (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## агентမှတ်ဉာဏ်အမျိုးအစားများ

ဒီမှတ်စုစာအုပ်မှာ Lesson 13 မှအဓိကမှတ်စုစာအုပ်ကနေ တူညီတဲ့သုံးမျိုးမှတ်ဉာဏ်အမျိုးအစားတွေကို စမ်းသပ်ပေးထားပြီး Cognee ကို ရေရှည်မှတ်ဉာဏ် backend အဖြစ် အသုံးပြုထားပါတယ်။

| မှတ်ဉာဏ်အမျိုးအစား | လုပ်ဆောင်ပုံ | အသက်တာ |
|---|---|---|
| **အလုပ်လုပ်နေသော** | `agent.create_session()` (MAF) | တစ်ခုပြောဆိုမှု |
| **တိုတောင်းသည့်** | Cognee session cache (Redis) | တစ်ခုပြောဆိုမှု session |
| **ရေရှည်သည့်** | Cognee သိပ္ပံပုံနှင့် vectors | အမြဲတမ်း |

### Cognee ၏ မှတ်ဉာဏ်ဆောက်လုပ်ပုံ
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Cognee သိမ်းဆည်းမှုကို ပြင်ဆင်ရန်


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## အပိုင်း ၁ — သိပ္ပံအခြေခံ ထူဆောက်ခြင်း

ကျွန်ုပ်တို့သည် ကုဒ်ရေးသားသူ အကူအညီဆောင်အတွက် ကျယ်ပြန့်သော သိပ္ပံအခြေခံကို ဖန်တီးရန် အချက်အလက် အမျိုးအစားသုံးမျိုးကို စုဆောင်းသည်။

၁။ **ဖန်တီးသူ ကိုယ်ရေးအကျဉ်း** — ကိုယ်ပိုင် ကျွမ်းကျင်မှုနှင့် နည်းပညာ နောက်ခံ
၂။ **Python အကောင်းဆုံး လေ့လာမှုများ** — Python ၏ ဇိမ့်ဓလေ့နှင့် လက်တွေ့ လမ်းညွှန်ချက်များ
၃။ **သမိုင်းအစ conversation များ** — ဖန်တီးသူများနှင့် AI အကူအညီ ပေးသူများအကြား ယခင် မေးမြန်းမှုနှင့် ဖြေရှင်းမှုများ


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## အသိပညာဂရပ်ကို မြင်ကွင်းတင်ပြပါ

Cognee သည် ထုတ်ယူထားသော အရာရှိများနှင့် ဆက်သွယ်မှုများ၏ အပြန်အလှန် HTML မြင်ကွင်းတင်ပြမှုကို ရုပ်သိမ်းနိုင်သည်။


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Memify ဖြင့် မှတ်ဉာဏ်အား တိုးတက်စေခြင်း

`memify()` သည် သိမြင်မှုဇယားကို ချဲ့ထွင်၍ သိပ္ပံနည်းစည်းမျဉ်းများ ဖန်တီးပေးသည် — နမူနာများ၊ အကောင်းဆုံးနည်းလမ်းများနှင့် သဘောတူညီချက်များအကြား ဆက်နွယ်မှုများကို သတ်မှတ်ပေးသည်။


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## အပိုင်း ၂ — Cognee ကိရိယာများနှင့်အတူ MAF အေးဂျင့်

ယခုတွင် `@tool` ဖန်ကာရှင်းများမှတဆင့် Cognee ၏ သိမြင်ခွင့်ဂရပ်ကို မေးမြန်းနိုင်သော MAF အေးဂျင့်တစ်ခုကို ဖန်တီးမှာဖြစ်သည်။ ၎င်းသည် အေးဂျင့်ကို ဂရပ်အသိနားလည်မှုရှိသော စာလုံးရေစစ်ဆေးမှု၏ အင်အားပြည့်ဝမှုကို အသုံးပြုခွင့်ပြုပြီး ဆက်သွယ်ဆွေးနွေးမှု သဘောတရားကို ဆက်လက် ဂရမ်စောင့်ရှောက်ခြင်းဖြင့် ထိန်းသိမ်းနိုင်သည်။


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## ဆက်စပ်အမှတ်အသားများဖြင့် အလုပ်လုပ်သော မှတ်ဉာဏ်

`AgentSession` ( `agent.create_session()` မှတည်ဆောက်ထားသည်) သည် ဆက်စပ်အမှတ်အသားအတွင်း အလုပ်လုပ်သော မှတ်ဉာဏ်ကို ပံ့ပိုးပေးသည်။ agent သည် ယခင်မက်ဆေ့ချ်များကို ပြန်လည်အပေါ်အောက် စစ်ဆေးနိုင်ပြီး ကျောင်းနနည်း၏ ကြာရှည်သိပ္ပံပညာဇယားကိုလည်း ရှာဖွေရန် ပြုလုပ်နိုင်သည်။


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## အစစ်အစဉ် အသစ် — အတက်မပြတ် မှတ်ဉာဏ် တည်ရှိနေသည်

အသစ်စက်စက် အစစ်အစဉ်တစ်ခုကို စတင်သောအခါ လက်ရှိ အသုံးပြုနေသော မှတ်ဉာဏ်သည် ဖယ်ရှားသွားပေမယ့်၊ Cognee နည်းပညာ ဂရပ်သည် မသေချာစွာ ရရှိနေနိုင်သည်။ အေးဂျင့်သည် အလုံးစုံအသစ်သော ဆက်သွယ်မှုတွင် တူညီသော အတက်မပြတ် သိပ္ပံပညာကို ပြန်လည်ရယူနိုင်သည်။


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## အကျဉ်းချုပ်

ဒီ notebook မှာ သင်က **MAF ရဲ့ အလုပ်လုပ်နိုင်သောမွတ်ဉာဏ်** (`agent.create_session()`) နဲ့ **Cognee ရဲ့ ရေရှည် အသိပညာဇယား** ကို ပေါင်းစပ်ထားတဲ့ ကုဒ်ရေးသူ အကူအညီကို တည်ဆောက်ခဲ့တယ်။

### သင်ယူသည့် အချက်များ
1. **အသိပညာဇယား တည်ဆောက်ခြင်း**: Cognee သည် ဖွဲ့စည်းမထားသော စာသားများကို စုပ်ယူပြီး ဇယားနှင့် vector memory တစ်ခု တည်ဆောက်ပါတယ်။
2. **memify ဖြင့် ဇယားတိုးချဲ့ခြင်း**: ရှိပြီးသား ဇယားအပေါ် မှတ်ချက်များနှင့် ပိုမိုစွမ်းဆောင်ရည်ရှိသော ဆက်ဆံရေးများ ထပ်ဖြည့်ချက်။
3. **MAF + Cognee ပေါင်းစပ်ခြင်း**: `@tool` လုပ်ဆောင်ချက်များက MAF agent ကို Cognee ရဲ့ ဇယားကို သဘာဝကျကျ မေးမြန်းခွင့်ပြုတယ်။
4. **အလုပ်လုပ်နိုင်သော မွတ်ဉာဏ် + ရေရှည်မွတ်ဉာဏ်**: `AgentSession` (via `agent.create_session()`) က session context ပေးပြီး Cognee က တည်ငြိမ်သော အသိပညာ ပေးပါတယ်။
5. **NodeSets ဖြင့် စစ်ထုတ်ရွေးချယ်ခြင်း**: အသိပညာဇယားက အထူးသတ်မှတ်ထားသော အစိတ်အပိုင်းများ (ဥပမာ- တည်ဆောင်ချက်များသာ) ကို ရည်ရွယ်သည်။

### အဓိက သင်ခန်းစာများ
- **Cognee** က မူရင်းစာသားကို ဖွဲ့စည်းထားပြီး ဆက်ဆံရေးရှိသော မှတ်ဉာဏ်အဖြစ် ပြောင်းလဲတယ် — flat vector store ထက် ပို၍ စွမ်းအားရှိတယ်။
- **`@tool` လုပ်ဆောင်ချက်များ** က MAF agent များနဲ့ ပြင်ပ အသိပညာ စနစ်များကို တောက်လျှောက် ချိတ်ဆက်ပေးတယ်။
- **`AgentSession`** (via `agent.create_session()`) က ဆက်ဆံရေး တစ်ခုဆီစီအတွက် context ကို ရေရှည် မွတ်ဉာဏ်နဲ့ ခွဲခြားထားတယ်။
- တူညီသော အသိပညာဇယားသည် ရှေ့နေများနှင့် agent များစွာအတွက် ဝန်ဆောင်မှုပေးသည်။

### အကောင်းဆုံး အသုံးချမှုများ
- **Developer copilots**: ကုဒ်သုံးသပ်ခြင်း၊ ဖြစ်တတ်သော အခက်အခဲ ရှာဖွေရေး၊ မူပိုင်ခွင့် အကူအညီများ
- **Customer-facing copilots**: ထုတ်ကုန်စာရွက်များ၊ မေးမြန်းခွင့်များ (FAQs), CRM မှတ်တမ်းများအပေါ် ပံ့ပိုးကူညီသူများ
- **Internal expert copilots**: မူဝါဒ၊ ဥပဒေ သို့မဟုတ် လုံခြုံရေးအကူအညီများတောင် အတွင်းရေးပညာရှင်များ
- **သီးခြား ဒေတာအလွှာများ**: ဖွဲ့စည်းထားပြီး မဖွဲ့စည်းသေးသော ဒေတာများကို တစ်ခုတည်းသော စုံစမ်းမေးမြန်းနိုင်သော ဇယားဆိုင်ရာတွင် ပေါင်းစည်းခြင်း

### နောက်တစ်ဆင့်များ
- Cognee မှာ အချိန်အခြေပြု အသိပညာသိမြင်မှု စမ်းသပ်ရန်
- ကဏ္ဍအလိုက် ဇယားမှာ OWL ontology ကို သတ်မှတ်ရန်
- အချိန်အတိုင်းအတာများကို တိုးတက်စေရန် အသုံးပြုသူ တုံ့ပြန်မှု စနစ် ထည့်သွင်းရန်
- တူညီ Cognee မွတ်ဉာဏ် အလွှာမျှဝေနိုင်သော မျိုးစုံ agent စနစ်များသို့ တိုးချဲ့ရန်


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
